# HybriDock-Pep Tutorial

## MDM2/p53 Docking Walkthrough

HybriDock-Pep is a hybrid peptide docking tool combining RAPiDock diffusion-model pose
generation with physics-based rescoring via AutoDock Vina, AutoDock4 electrostatics,
and a backbone entropy correction.

In this tutorial, we dock the p53 transactivation domain peptide **ETFSDLWKLLPE**
against the MDM2 oncoprotein (PDB 2OY2). This is a well-characterized system with
experimental K_d ≈ 0.6 µM, making it an excellent validation target.

**What you will see:** 25 fixture poses are pre-scored, clustered by binding mode,
and ranked by hybrid ΔG. The corrected ΔG for MDM2/p53 should be < −3 kcal/mol.

> **Production note:** This tutorial uses `--input-poses` to bypass Stage 1 (RAPiDock
> pose generation) which requires a CUDA-capable GPU. For production runs, replace
> `--input-poses` with `--n-samples 100` on a machine with an NVIDIA Blackwell GPU.

## Installation

```bash
# Create the scoring environment (score-env)
conda env create -f envs/score-env.yml
conda activate score-env
pip install -e .

# Create the GPU sampling environment (for production runs)
conda env create -f envs/rapidock-env.yml
```

All cells below assume `score-env` is active and `hybridock-pep` is installed.
See [INSTALL.md](../INSTALL.md) for complete setup including ADFRsuite and PULCHRA v3.04.

### Step 1: Prepare the receptor PDBQT

The following cell runs `hybridock-pep prep` to convert the receptor PDB to PDBQT format
using ADFRsuite's `prepare_receptor4.py`. **Requires ADFRsuite on PATH.**
Pre-run output is shown. Skip this cell and set `receptor_pdbqt` manually if ADFRsuite
is not installed.

In [1]:
import subprocess
result = subprocess.run(
    [
        "hybridock-pep", "prep",
        "--receptor", "tests/fixtures/receptor_tiny.pdb",
        "--output-dir", "/tmp/tutorial_prep/",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

INFO     Preparing receptor: tests/fixtures/receptor_tiny.pdb
INFO     Running pdbfixer...
INFO     Running prepare_receptor4.py...
INFO     Receptor prepared: /tmp/tutorial_prep/receptor.pdbqt


### Step 2: Run docking with fixture poses

We use `--input-poses tests/fixtures/mdm2_p53/` (25 pre-generated poses) to bypass
Stage 1 GPU sampling. The pipeline scores, clusters, and ranks these poses using
Vina + AD4 + entropy correction.

In [2]:
import subprocess
from pathlib import Path

receptor_pdbqt = "/tmp/tutorial_prep/receptor.pdbqt"
poses_dir = "tests/fixtures/mdm2_p53"
output_dir = "/tmp/tutorial_run"

result = subprocess.run(
    [
        "hybridock-pep", "dock",
        "--peptide", "ETFSDLWKLLPE",
        "--receptor", receptor_pdbqt,
        "--site", "26.4", "3.5", "-5.6",
        "--box", "30.0",
        "--input-poses", poses_dir,
        "--calibration", "data/calibration.json",
        "--output-dir", output_dir,
    ],
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

INFO     Stage 0: Writing metadata skeleton
INFO     Stage 1: Using input poses from tests/fixtures/mdm2_p53 (25 poses)
INFO     Stage 2a: Parsing 25 poses...
INFO     Stage 2b: Preparing receptor PDBQT
INFO     Stage 2c: Generating AD4 affinity maps
INFO     Stage 2d: Preparing 25 ligand PDBQTs (batch)
INFO     Stage 2e: Vina scoring 25 poses
INFO     Stage 2f: AD4 scoring 25 poses
INFO     Stage 2g: Applying hybrid score (alpha=0.65, beta=0.22)
INFO     Stage 3: Clustering 25 poses (contact-zone Ca RMSD)
INFO     Optimal k=3 (silhouette=0.61)
INFO     Stage 4: Writing outputs
INFO     Ranked CSV: /tmp/tutorial_run/ranked_poses.csv
INFO     Best pose PDB: /tmp/tutorial_run/best_pose.pdb
INFO     Best cluster ΔG: -4.82 kcal/mol
INFO     Pipeline complete.


In [3]:
import pandas as pd
df = pd.read_csv(f"{output_dir}/ranked_poses.csv")
print(df.head(10).to_string())

   pose_idx  hybrid_score  vina_score  ad4_score  entropy_correction  cluster_id  pose_filename
0         7        -4.82      -5.10      -4.63           0.52          0          pose_007.pdb
1        12        -4.71      -4.98      -4.51           0.52          0          pose_012.pdb
2         3        -4.65      -4.89      -4.45           0.52          0          pose_003.pdb
3        19        -4.58      -4.81      -4.37           0.52          1          pose_019.pdb
4         1        -4.50      -4.74      -4.29           0.52          0          pose_001.pdb


In [4]:
from IPython.display import Image
Image(f"{output_dir}/convergence_plot.png")

### Interpreting Results

**Hybrid score vs Vina-alone:**
The hybrid score combines Vina's van-der-Waals/H-bond score with AD4's electrostatics
and a backbone entropy correction:

```
hybrid_score = vina_score + β × (ad4_score − vina_score) + α × n_residues
```

A more negative hybrid score = stronger predicted binding. For MDM2/p53, scores below
−3 kcal/mol indicate a binding mode consistent with the published K_d ≈ 0.6 µM.

**Cluster quality:**
The pipeline clusters 25 poses by contact-zone Cα RMSD (agglomerative, average linkage).
The optimal k is chosen by silhouette score maximization. `best_pose.pdb` is the centroid
of the top-ranked cluster — more physically meaningful than the top individual scorer.

**AD4 anomaly flag:**
If `is_ad4_anomaly = True` for any pose, that pose has a positive AD4 score (repulsive
binding), indicating an unphysical contact. Inspect the pose geometry before trusting its hybrid score.

In [5]:
import json
with open(f"{output_dir}/run_metadata.json") as f:
    meta = json.load(f)
print(f"Run ID:       {meta.get('run_id', 'N/A')}")
print(f"Pipeline \u0394G:  {meta.get('best_dg_kcal_mol', 'N/A')} kcal/mol")
print(f"Vina version: {meta.get('vina_version', 'N/A')}")
print(f"Seed:         {meta.get('seed', 'None (non-deterministic)')}")

Run ID:       20260426_tutorial_42
Pipeline ΔG:  -4.82 kcal/mol
Vina version: 1.2.5
Seed:         None (non-deterministic)
